# Optional Bonus · AI/BI Dashboards and Genie

This notebook is an optional bonus after the main Day 2 path. It is useful when the class has time for Databricks-native BI beyond the required Power BI and performance modules.

It reuses the same governed Gold and W2 serving objects that Power BI uses, but it is not required to complete the core training flow.


## Business Scenario

RetailHub's Sales Director wants a governed dashboard and a conversational interface for common sales questions. If time permits, the data team can show how AI/BI complements Power BI and how to prepare a safe semantic context for Genie.


## Learning Objectives

By the end of this optional bonus you can:

- compare AI/BI Dashboards and Power BI,
- define Genie Space and Verified Answers,
- create dashboard dataset queries from governed Gold objects,
- prepare semantic context and business instructions,
- identify current governance and limitation considerations.


## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `gold.fact_sales_dashboard`.


In [ ]:
%run ../../setup/00_setup


### Configuration

Confirm the active catalog and schemas before running the Day 2 examples.


In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))


### Runtime Pre-check

This optional bonus requires the Workshop 2 dataset/performance layer. If the check fails, skip this notebook or run `solution/w2_powerbi_dataset_solution.ipynb` first.


In [ ]:
required_tables = [
    f"{GOLD}.fact_sales_dashboard_monthly",
    f"{GOLD}.v_fact_sales_incremental",
    f"{GOLD}.kpi_monthly",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Complete Workshop 2 before running this notebook.")
print("[OK] Workshop 2 serving objects are available.")


## 1. AI/BI Dashboard vs Power BI

An **AI/BI Dashboard** is a Databricks-native dashboard built directly on governed lakehouse data. It is useful when teams want analytics close to SQL Warehouse, Unity Catalog and Databricks users.

A **Power BI semantic model** is still the better default when the organization needs enterprise distribution, reusable DAX measures, certified datasets and rich report design. Treat AI/BI as a governed lakehouse-native consumption path, not as a replacement for every Power BI report.

Expected result: the right tool depends on ownership, audience, latency needs and governance model. Both tools should reuse the same trusted Gold and W2 objects.


In [ ]:
%sql
WITH decision AS (
  SELECT 'AI/BI Dashboard' AS tool, 'Databricks-native operational analytics on governed Gold data' AS best_for, 'less Power BI-specific report design' AS tradeoff
  UNION ALL SELECT 'Power BI', 'enterprise semantic models and broad business distribution', 'requires connector, refresh and separate model governance'
  UNION ALL SELECT 'Both', 'shared Gold source with different consumption experiences', 'requires clear KPI ownership'
)
SELECT *
FROM decision


## 2. AI/BI Serving Dataset

A **serving dataset** is the table or view that a BI surface reads directly. For AI/BI and Genie it should have business-friendly names, documented grain and no unnecessary technical columns.

Create one durable AI/BI summary object because dashboards and Genie need a stable governed source. This object is intentionally small: it reuses the W2 monthly aggregate and does not rebuild the Day 1 Gold model.


In [ ]:
%sql
CREATE OR REPLACE TABLE gold.fact_sales_aibi_summary
COMMENT 'AI/BI serving table for Sales dashboard and Genie: monthly sales KPIs by region, category and channel.'
AS
SELECT
  year_month,
  order_month,
  customer_region,
  category,
  channel,
  SUM(completed_orders) AS completed_orders,
  ROUND(SUM(revenue), 2) AS revenue,
  ROUND(SUM(gross_margin), 2) AS gross_margin,
  ROUND(100.0 * SUM(gross_margin) / NULLIF(SUM(revenue), 0), 2) AS margin_rate_pct
FROM gold.fact_sales_dashboard_monthly
GROUP BY year_month, order_month, customer_region, category, channel


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT concat_ws('|', year_month, customer_region, category, channel)) AS distinct_grain_keys,
  MIN(year_month) AS min_month,
  MAX(year_month) AS max_month
FROM gold.fact_sales_aibi_summary


## 3. Dashboard Dataset Queries

A **dashboard dataset query** is the SQL result set used by one or more visualizations. Keep each dataset query narrow, filtered where possible and aligned with the visual grain.

Validate the exact SQL in the notebook before pasting it into the AI/BI Dashboard UI. This keeps the workflow reproducible even when the final dashboard is created through the workspace interface.


### Dataset 1: Revenue Trend by Channel


In [ ]:
%sql
SELECT
  year_month,
  channel,
  ROUND(SUM(revenue), 2) AS revenue
FROM gold.fact_sales_aibi_summary
GROUP BY year_month, channel
ORDER BY year_month, channel


### Dataset 2: Margin by Category and Channel

Ratios must be re-aggregated carefully. Use revenue-weighted margin, not a simple average of percentages.


In [ ]:
%sql
SELECT
  category,
  channel,
  ROUND(100.0 * SUM(gross_margin) / NULLIF(SUM(revenue), 0), 2) AS margin_rate_pct,
  ROUND(SUM(revenue), 2) AS revenue
FROM gold.fact_sales_aibi_summary
GROUP BY category, channel
ORDER BY revenue DESC


### Dataset 3: KPI Cards


In [ ]:
%sql
SELECT
  MAX(year_month) AS latest_month,
  ROUND(SUM(revenue), 2) AS revenue,
  ROUND(SUM(gross_margin), 2) AS gross_margin,
  SUM(completed_orders) AS completed_orders,
  ROUND(100.0 * SUM(gross_margin) / NULLIF(SUM(revenue), 0), 2) AS margin_rate_pct
FROM gold.fact_sales_aibi_summary


## 4. UI Checklist: Build AI/BI Dashboard

1. Open Databricks SQL or AI/BI Dashboards.
2. Create a new dashboard named `RetailHub Sales Executive`.
3. Add dataset queries from this notebook.
4. Create one line chart, one grouped bar chart and KPI counters.
5. Confirm each visualization uses `gold.fact_sales_aibi_summary` or `gold.kpi_monthly`.
6. Share only with users allowed to read the Gold schema.


## 5. Genie Space Definition

A **Genie Space** is a governed conversational BI experience over selected Databricks data. It needs clear tables, trusted measures, business instructions and examples of good questions.

A **semantic context** is the business explanation attached to the data context: table purpose, KPI definitions, synonyms, allowed interpretations and caveats. Without it, the same column names can be interpreted differently by different users.

A **Verified Answer** is a curated response pattern for a known business question. Use it to anchor important KPI definitions, reduce ambiguity and teach Genie how the organization expects common questions to be answered.


### Genie Semantic Context

Recommended Genie tables:

| Table | Purpose |
|---|---|
| `gold.fact_sales_aibi_summary` | monthly revenue, margin and order trends |
| `gold.kpi_monthly` | KPI card values and month-level sanity checks |
| `gold.dim_customer` | region and segment vocabulary |
| `gold.dim_product` | category and product vocabulary |

Business instructions:

- Revenue means completed sales revenue.
- Margin rate is gross margin divided by revenue.
- Use `year_month` for monthly trend questions.
- If the user asks for live line-level detail, explain that the dashboard summary is monthly and direct them to `gold.v_fact_sales_incremental`.


In [ ]:
%sql
SELECT
  'What was revenue by channel last month?' AS sample_question,
  'Use gold.fact_sales_aibi_summary, filter to latest year_month, group by channel, sum revenue.' AS expected_logic
UNION ALL
SELECT
  'Which category has the best margin rate?',
  'Use revenue-weighted margin: SUM(gross_margin) / SUM(revenue), grouped by category.'
UNION ALL
SELECT
  'Show monthly revenue trend for a region.',
  'Filter customer_region, group by year_month, sum revenue.'


## 6. UI Checklist: Configure Genie Space

1. Create a Genie Space for RetailHub Sales.
2. Add the recommended Gold tables.
3. Paste the business instructions from this notebook.
4. Add sample questions as Verified Answers where available.
5. Test ambiguous prompts such as `margin` and `orders`.
6. Confirm permissions are inherited from Unity Catalog.


## 7. Limitations and Governance

Genie does not remove the need for a governed data model. It amplifies the model quality: good table names, comments, measures and instructions produce better answers.

A **governance boundary** is the line between what the AI/BI experience may answer and what must be escalated to a certified report, a data owner or a separate access-controlled dataset. Define that boundary before inviting broad business usage.

Governance checklist:

- Gold schema permissions are correct,
- KPI definitions are documented,
- sample questions cover the most common executive asks,
- sensitive fields are excluded or masked,
- workspace users understand that Genie answers depend on selected data context.


## 8. AI/BI and Power BI Together

AI/BI and Power BI should reuse the same Gold and W2 serving objects. Power BI can remain the enterprise reporting layer, while AI/BI supports Databricks-native exploration and rapid governed dashboards.


## Summary

This optional bonus closes with a Databricks-native consumption path over the same governed data. The required Day 2 flow can finish without this notebook; use it when the class has time for AI/BI Dashboards and Genie.
